<a href="https://colab.research.google.com/github/Jesus-0655/Satelites_Actividad_IA_LITIID74/blob/main/AstroShield_LITIID74_Equipo_Dinamita.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## AstroShield — Motor Lógico y Optimización de la Constelación
Aqui esta toda la lógica matemática del modelo planetario, el sistema de coordenadas,
el cálculo de cobertura por muestreo, el mecanismo de reparación de restricciones y el algoritmo evolutivo completo.
Jesus.

In [ ]:
import numpy as np
import math
import time
import json

# --- CONSTANTES GLOBALES Y MODELO ---
R_EARTH = 6371.0                  # Radio de la Tierra (km)
SAT_FOOTPRINT_RADIUS_KM = 1490.0  # Radio de la huella física proyectada de cada satélite (km, ≈13.4° de arco)
REGION_RADIUS_KM = 300.0          # Radio de observación de cada región alrededor de su centro (km)
LIMIT_SEPARATION_DEG = 5.0        # Separación mínima entre satélites

# --- DATOS DE REGIONES DE INTERÉS ---
REGIONS = [
    {"name": "Península de Yucatán, México", "lat": 20.5, "lon": -89.0, "priority_weight": 3},
    {"name": "Amazonía brasileña", "lat": -3.5, "lon": -62.0, "priority_weight": 3},
    {"name": "California, Estados Unidos", "lat": 36.8, "lon": -119.4, "priority_weight": 2},
    {"name": "Región del Sahel, África", "lat": 15.0, "lon": 0.0, "priority_weight": 2},
    {"name": "Japón", "lat": 36.0, "lon": 138.0, "priority_weight": 2},
    {"name": "Interior de Australia", "lat": -25.0, "lon": 133.0, "priority_weight": 1}
]

# --- RESTRICCIONES ESPACIALES ---
FORBIDDEN_ZONES = [
    # Zonas de desechos espaciales
    {"name": "Atlántico Norte", "lat": 45.0, "lon": -35.0, "radius": 12.0},
    {"name": "Pacífico Sur", "lat": -30.0, "lon": -120.0, "radius": 10.0},
    {"name": "Región polar norte", "lat": 78.0, "lon": 30.0, "radius": 8.0},
    # Posiciones ocupadas
    {"name": "Objeto A", "lat": 10.0, "lon": 45.0, "radius": 5.0},
    {"name": "Objeto B", "lat": -15.0, "lon": 110.0, "radius": 5.0}
]

# --- FUNCIONES GEOMÉTRICAS ---
def haversine_distance_deg(lat1, lon1, lat2, lon2):
    """Distancia angular (grados) entre dos coordenadas esféricas."""
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    delta_phi = math.radians(lat2 - lat1)
    delta_lambda = math.radians(lon2 - lon1)
    a = math.sin(delta_phi/2.0)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(delta_lambda/2.0)**2
    c = 2.0 * math.atan2(math.sqrt(a), math.sqrt(1.0 - a))
    return math.degrees(c)

def great_circle_km(lat1, lon1, lat2, lon2):
    """Distancia ortodrómica real en kilómetros sobre la superficie terrestre."""
    return R_EARTH * math.radians(haversine_distance_deg(lat1, lon1, lat2, lon2))

def wrap_longitude(lon):
    """Ajusta la longitud al rango [-180, 180]."""
    return (lon + 180.0) % 360.0 - 180.0

def sample_sphere_point_uniform():
    """Punto uniforme en la esfera (arcsin evita acumulación en polos)."""
    lon = np.random.uniform(-180.0, 180.0)
    lat = math.degrees(math.asin(np.random.uniform(-1.0, 1.0)))
    return lat, lon

# Implementar validación y reparación de posición en zonas restringidas
Julian.

In [ ]:
def is_inside_restricted_zone(lat, lon):
    """Verifica si un punto cae dentro de alguna zona restringida u objeto ocupado."""
    for zone in FORBIDDEN_ZONES:
        dist = haversine_distance_deg(lat, lon, zone["lat"], zone["lon"])
        if dist <= zone["radius"]:
            return True, zone, dist
    return False, None, 0.0

def repair_satellite(lat, lon):
    """Aplica el algoritmo de reparación para empujar el satélite hacia los bordes."""
    current_lat, current_lon = lat, lon
    for attempt in range(10):
        inside, zone, dist = is_inside_restricted_zone(current_lat, current_lon)
        if not inside:
            return current_lat, current_lon

        # Empujar a lo largo del círculo máximo alejándolo del centro de la zona
        d_lat = current_lat - zone["lat"]
        d_lon = wrap_longitude(current_lon - zone["lon"])
        norm = math.sqrt(d_lat**2 + d_lon**2)
        if norm < 1e-6:
            # Si coincide exactamente con el centro, aplicamos un empuje aleatorio
            d_lat, d_lon = np.random.uniform(-1, 1), np.random.uniform(-1, 1)
            norm = math.sqrt(d_lat**2 + d_lon**2)

        # Mover al límite de la zona + 0.5 grados de margen
        push_factor = (zone["radius"] + 0.5) / (norm if norm > 0 else 1.0)
        current_lat = np.clip(zone["lat"] + d_lat * push_factor, -90.0, 90.0)
        current_lon = wrap_longitude(zone["lon"] + d_lon * push_factor)

    # Si falla 10 veces, se reubica totalmente al azar de forma uniforme
    return sample_sphere_point_uniform()

def process_and_repair_individual(coords):
    """Repara cada satélite de forma individual e independiente en un cromosoma (8 pares)."""
    repaired = []
    for i in range(8):
        lat, lon = coords[2*i], coords[2*i+1]
        rep_lat, rep_lon = repair_satellite(lat, lon)
        repaired.extend([rep_lat, rep_lon])
    return repaired

```markdown
# Evaluación Fitness
Ivan

In [ ]:
def region_coverage_status(sats):
    """
    Determina, en kilómetros, si cada región está cubierta.

    Una región (disco de REGION_RADIUS_KM alrededor de su centro) está CUBIERTA cuando al menos
    un satélite contiene todo el disco dentro de su huella: dos casquetes esféricos, uno dentro
    del otro, cumplen  d(satélite, centro) + radio_región <= radio_huella.
    (Con REGION_RADIUS_KM = 0 el criterio se reduce a "el centro está dentro de la huella").
    """
    max_center_distance_km = SAT_FOOTPRINT_RADIUS_KM - REGION_RADIUS_KM
    status = []
    for r in REGIONS:
        nearest_km = min(great_circle_km(s_lat, s_lon, r["lat"], r["lon"]) for s_lat, s_lon in sats)
        status.append({
            "name": r["name"],
            "covered": nearest_km <= max_center_distance_km,
            "nearest_km": nearest_km
        })
    return status

def evaluate_configuration(coords):
    """
    Evalúa la aptitud (Fitness) y verifica la validez según restricciones físicas y de proximidad.
    Fitness = cobertura ponderada: suma de prioridades de las regiones cubiertas / suma total de prioridades.
    Retorna: (fitness, is_valid, total_violation_degrees)
    """
    sats = [(coords[2*i], coords[2*i+1]) for i in range(8)]

    is_valid = True
    total_violation = 0.0

    # Límites de coordenadas
    for lat, lon in sats:
        if not (-90.0 <= lat <= 90.0 and -180.0 <= lon <= 180.0):
            is_valid = False
            total_violation += 999.0

    # Distancia mínima entre satélites
    for i in range(8):
        for j in range(i+1, 8):
            dist = haversine_distance_deg(sats[i][0], sats[i][1], sats[j][0], sats[j][1])
            if dist < LIMIT_SEPARATION_DEG:
                is_valid = False
                total_violation += (LIMIT_SEPARATION_DEG - dist)

    # Colisiones con zonas restringidas
    for lat, lon in sats:
        inside, zone, dist = is_inside_restricted_zone(lat, lon)
        if inside:
            is_valid = False
            total_violation += (zone["radius"] - dist)

    # Cobertura ponderada (el peso total se recalcula siempre: sigue válido si cambian las prioridades)
    total_weight = sum(r["priority_weight"] for r in REGIONS)
    status = region_coverage_status(sats)
    covered_weight = sum(r["priority_weight"] for r, st in zip(REGIONS, status) if st["covered"])

    fitness = covered_weight / total_weight
    return fitness, is_valid, total_violation


   # Algoritmo Evolutivo AstroShield con reparación, factibilidad, selección por torneo y elitismo.
    Retorna: (best_fitness, best_individual, is_valid, total_violation, history)
    Fernando

In [ ]:
def genetic_algorithm(pop_size=100, max_gens=200, sigma=3.0, p_mut=0.15, random_state=None):
    """Algoritmo Evolutivo AstroShield con reparación, factibilidad, selección por torneo y elitismo.
    Retorna: (best_fitness, best_individual, is_valid, total_violation, history)
    """
    if random_state is not None:
        np.random.seed(random_state)

    # Inicialización de la población utilizando muestreo uniforme real en esfera
    population = []
    for _ in range(pop_size):
        ind = []
        for _ in range(8):
            lat, lon = sample_sphere_point_uniform()
            ind.extend([lat, lon])
        population.append(process_and_repair_individual(ind))

    best_ever_individual = None
    best_ever_fitness = -1.0
    best_ever_valid = False
    best_ever_violation = float('inf')

    generations_no_improvement = 0
    history = []

    for gen in range(max_gens):
        # 1. Evaluación completa de la generación actual
        evaluated_pop = []
        for ind in population:
            fit, is_val, viol = evaluate_configuration(ind)
            evaluated_pop.append({"coords": ind, "fitness": fit, "valid": is_val, "violation": viol})

        # Función clave para ordenar según Factibilidad
        def feasibility_key(ind_dict):
            if ind_dict["valid"]:
                return (1, ind_dict["fitness"], 0.0)
            else:
                return (0, 0.0, -ind_dict["violation"])

        evaluated_pop.sort(key=feasibility_key, reverse=True)

        # Registro de métricas estadísticas de la generación
        gen_fitnesses = [ind["fitness"] for ind in evaluated_pop if ind["valid"]]
        if not gen_fitnesses:
            # Si no hay válidos, registramos 0
            gen_fitnesses = [0.0]

        history.append({
            "generation": gen,
            "best": evaluated_pop[0]["fitness"] if evaluated_pop[0]["valid"] else 0.0,
            "mean": float(np.mean(gen_fitnesses)),
            "worst": float(np.min(gen_fitnesses))
        })

        # Monitorear al mejor histórico de esta corrida
        current_best = evaluated_pop[0]
        improved = False
        if best_ever_individual is None:
            improved = True
        else:
            if feasibility_key(current_best) > feasibility_key({"coords": best_ever_individual, "fitness": best_ever_fitness, "valid": best_ever_valid, "violation": best_ever_violation}):
                if best_ever_valid and current_best["valid"]:
                    if (current_best["fitness"] - best_ever_fitness) > 1e-4:
                        improved = True
                else:
                    improved = True

        if improved:
            best_ever_individual = list(current_best["coords"])
            best_ever_fitness = current_best["fitness"]
            best_ever_valid = current_best["valid"]
            best_ever_violation = current_best["violation"]
            generations_no_improvement = 0
        else:
            generations_no_improvement += 1

        # Parada anticipada
        if generations_no_improvement >= 40:
            break

        # 2. Elitismo Estricto
        next_population = [evaluated_pop[0]["coords"], evaluated_pop[1]["coords"]]

        # 3. Torneo de Selección (k = 3)
        def tournament_selection(evaluated_list):
            candidates = np.random.choice(evaluated_list, size=3, replace=False)
            candidates = list(candidates)
            candidates.sort(key=feasibility_key, reverse=True)
            return candidates[0]["coords"]

        # 4. Cruzamiento y Mutación
        while len(next_population) < pop_size:
            parent1 = tournament_selection(evaluated_pop)
            parent2 = tournament_selection(evaluated_pop)

            if np.random.rand() < 0.90:
                child = []
                for s in range(8):
                    if np.random.rand() < 0.5:
                        child.extend(parent1[2*s : 2*s+2])
                    else:
                        child.extend(parent2[2*s : 2*s+2])
            else:
                child = list(parent1)

            for s in range(8):
                if np.random.rand() < p_mut:
                    lat_noise = np.random.normal(0, sigma)
                    lon_noise = np.random.normal(0, sigma)
                    child[2*s] = np.clip(child[2*s] + lat_noise, -90.0, 90.0)
                    child[2*s+1] = wrap_longitude(child[2*s+1] + lon_noise)

                if np.random.rand() < 0.02:
                    u_lat, u_lon = sample_sphere_point_uniform()
                    child[2*s] = u_lat
                    child[2*s+1] = u_lon

            repaired_child = process_and_repair_individual(child)
            next_population.append(repaired_child)

        population = next_population

    return best_ever_fitness, best_ever_individual, best_ever_valid, best_ever_violation, history

Barrido de hiperparámetros

Dante

In [ ]:
# --- BARRIDO DE HIPERPARAMETROS (GRID SWEEP) ---
# Parametros a probar
pop_sizes = [50, 100, 200]
sigmas = [1.0, 3.0, 6.0]
p_mutations = [0.05, 0.15, 0.30]

# 10 Semillas constantes de prueba
seeds = [42, 101, 2023, 777, 8888, 12, 999, 1312, 456, 88]

best_overall_score = -1.0
best_config = {}

print("Iniciando barrido de hiperparametros de optimizacion...")
print(f"Total de combinaciones a evaluar: {len(pop_sizes)*len(sigmas)*len(p_mutations)} con 10 iteraciones por caso.")

t0 = time.time()

for pop in pop_sizes:
    for sig in sigmas:
        for p_mut in p_mutations:
            scores = []
            val_count = 0
            for seed in seeds:
                # Se desempaquetan los 5 valores retornados por genetic_algorithm
                fit, _, val, _, _ = genetic_algorithm(pop_size=pop, max_gens=100, sigma=sig, p_mut=p_mut, random_state=seed)
                if val:
                    scores.append(fit)
                    val_count += 1
                else:
                    scores.append(0.0) # Se penaliza a 0 si no genera solucion valida

            avg_score = np.mean(scores)
            print(f"Pob: {pop} | Sigma: {sig} | P.Mut: {p_mut} -> Fitness Promedio: {avg_score:.5f} (Validos: {val_count}/10)")

            if avg_score > best_overall_score:
                best_overall_score = avg_score
                best_config = {"pop_size": pop, "sigma": sig, "p_mut": p_mut}

t_end = time.time()
print("\n--- BARRIDO COMPLETADO ---")
print(f"Tiempo total de computo: {t_end - t0:.2f} segundos")
print(f"La mejor combinacion encontrada es:")
print(f" - Tamano de Poblacion: {best_config['pop_size']}")
print(f" - Sigma (Ruido Gaussiano): {best_config['sigma']}")
print(f" - Probabilidad de mutacion por satelite: {best_config['p_mut']}")
print(f" - Fitness promedio obtenido: {best_overall_score:.5f}")

Iniciando barrido de hiperparametros de optimizacion...
Total de combinaciones a evaluar: 27 con 10 iteraciones por caso.
Pob: 50 | Sigma: 1.0 | P.Mut: 0.05 -> Fitness Promedio: 0.88000 (Validos: 10/10)
Pob: 50 | Sigma: 1.0 | P.Mut: 0.15 -> Fitness Promedio: 0.87231 (Validos: 10/10)
Pob: 50 | Sigma: 1.0 | P.Mut: 0.3 -> Fitness Promedio: 0.87538 (Validos: 10/10)
Pob: 50 | Sigma: 3.0 | P.Mut: 0.05 -> Fitness Promedio: 0.88000 (Validos: 10/10)
Pob: 50 | Sigma: 3.0 | P.Mut: 0.15 -> Fitness Promedio: 0.88308 (Validos: 10/10)
Pob: 50 | Sigma: 3.0 | P.Mut: 0.3 -> Fitness Promedio: 0.87538 (Validos: 10/10)
Pob: 50 | Sigma: 6.0 | P.Mut: 0.05 -> Fitness Promedio: 0.87385 (Validos: 10/10)
Pob: 50 | Sigma: 6.0 | P.Mut: 0.15 -> Fitness Promedio: 0.87692 (Validos: 10/10)
Pob: 50 | Sigma: 6.0 | P.Mut: 0.3 -> Fitness Promedio: 0.86308 (Validos: 10/10)
Pob: 100 | Sigma: 1.0 | P.Mut: 0.05 -> Fitness Promedio: 0.88615 (Validos: 10/10)
Pob: 100 | Sigma: 1.0 | P.Mut: 0.15 -> Fitness Promedio: 0.89077 (Vali